In [1]:
import pandas as pd
import numpy as np
import os
import gc
import random
import warnings
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import KFold

warnings.filterwarnings('ignore')

# ==============================================================================
# ★ ここで切り替え ★
# ==============================================================================
# 'GRU'  -> 高精度が期待できる (LSTMの軽量版)
# 'RNN'  -> 原始的。実験用 (勾配消失しやすい)
MODEL_TYPE = 'GRU' 
# MODEL_TYPE = 'RNN'

print(f">>> Running Mode: {MODEL_TYPE}")

# ==============================================================================
# 1. 設定
# ==============================================================================
if os.path.exists('/kaggle/input/joai-competition-2026/train.csv'):
    INPUT_DIR = '/kaggle/input/joai-competition-2026'
elif os.path.exists('/kaggle/input/ai2026/train.csv'):
    INPUT_DIR = '/kaggle/input/ai2026'
else:
    INPUT_DIR = '.' 

TRAIN_PATH = os.path.join(INPUT_DIR, 'train.csv')
TEST_PATH = os.path.join(INPUT_DIR, 'test.csv')

# ファイル名を自動設定
SUBMISSION_PATH = f'submission_{MODEL_TYPE.lower()}.csv'

# ハイパーパラメータ
MAX_LEN = 40
BATCH_SIZE = 64
N_FOLDS = 5
EPOCHS = 30
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using Device: {DEVICE}")

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)

# ==============================================================================
# 2. 前処理 (Time-Weighting + PID Features)
# ==============================================================================
# Ventilator-LSTMで成功したPID特徴量を今回も採用します
def add_features(df):
    df_eng = df.copy()
    ignore_cols = ['id', 'sample_id', 'day_n', 'time', 'lever', 'mouse_id']
    base_cols = [c for c in df_eng.columns if c not in ignore_cols]
    
    grouped = df_eng.groupby('sample_id')
    
    # Time Weighting Gain
    max_day = df_eng['day_n'].max()
    gain = 1.0 + (df_eng['day_n'] / max_day * 0.5)
    
    for col in base_cols:
        # Gain
        df_eng[f'{col}_w'] = df_eng[col] * gain
        # Diff (微分)
        df_eng[f'{col}_d1'] = grouped[col].diff().fillna(0) * gain
        # Cumsum (積分) - RNN/GRUは長期記憶が苦手なので、積分値を入れるのは効果絶大です
        df_eng[f'{col}_cum'] = grouped[col].cumsum()
        
    return df_eng

print(">>> Loading & Preprocessing...")
train = pd.read_csv(TRAIN_PATH).fillna(0)
test = pd.read_csv(TEST_PATH).fillna(0)

train = add_features(train)
test = add_features(test)

ignore_cols = ['id', 'sample_id', 'day_n', 'time', 'lever', 'mouse_id']
train_feats = [c for c in train.columns if c not in ignore_cols]
test_feats = set(test.columns)
feature_cols = [c for c in train_feats if c in test_feats]

scaler = RobustScaler()
train[feature_cols] = scaler.fit_transform(train[feature_cols])
test[feature_cols] = scaler.transform(test[feature_cols])

# ==============================================================================
# 3. Dataset
# ==============================================================================
class BrainDataset(Dataset):
    def __init__(self, df, feature_cols, target_col=None, max_len=40):
        self.df = df
        self.sample_ids = df['sample_id'].unique()
        self.feature_cols = feature_cols
        self.target_col = target_col
        self.max_len = max_len
        self.grouped = df.groupby('sample_id')

    def __len__(self):
        return len(self.sample_ids)

    def __getitem__(self, idx):
        sample_id = self.sample_ids[idx]
        group = self.grouped.get_group(sample_id)
        x = group[self.feature_cols].values
        seq_len = len(x)
        x_pad = np.zeros((self.max_len, len(self.feature_cols)), dtype=np.float32)
        x_pad[:seq_len, :] = x
        mask = np.zeros((self.max_len,), dtype=np.float32)
        mask[:seq_len] = 1.0
        ret = {'x': torch.tensor(x_pad), 'mask': torch.tensor(mask), 'id': sample_id}
        if self.target_col:
            y = group[self.target_col].values
            y_pad = np.zeros((self.max_len,), dtype=np.float32)
            y_pad[:seq_len] = y
            ret['y'] = torch.tensor(y_pad)
        return ret

# ==============================================================================
# 4. Model Definition (RNN / GRU)
# ==============================================================================
class ClassicRecurrentModel(nn.Module):
    def __init__(self, model_type, input_dim, hidden_dim=256, num_layers=3):
        super().__init__()
        
        self.model_type = model_type
        
        # Embedding
        self.embedding = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU()
        )
        
        # Main Recurrent Layer
        if model_type == 'GRU':
            self.rnn = nn.GRU(
                input_size=hidden_dim,
                hidden_size=hidden_dim,
                num_layers=num_layers,
                batch_first=True,
                bidirectional=True,
                dropout=0.1
            )
        elif model_type == 'RNN':
            # Vanilla RNN (non-linearity=tanh)
            self.rnn = nn.RNN(
                input_size=hidden_dim,
                hidden_size=hidden_dim,
                num_layers=num_layers,
                batch_first=True,
                bidirectional=True,
                dropout=0.1
            )
        else:
            raise ValueError("model_type must be 'GRU' or 'RNN'")

        # Regression Head
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_dim * 2),
            nn.Linear(hidden_dim * 2, 128),
            nn.GELU(),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x = self.embedding(x)
        # RNN/GRU output: (out, hidden_state)
        # out shape: [Batch, Time, Hidden*2]
        x, _ = self.rnn(x)
        out = self.head(x)
        return out.squeeze(-1)

# ==============================================================================
# 5. Training Loop
# ==============================================================================
final_test_preds = {} 
test_counts = {}

unique_samples = train['sample_id'].unique()
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
criterion = nn.MSELoss(reduction='none')

print(f"\n>>> Starting {N_FOLDS}-Fold CV with {MODEL_TYPE}...")

test_ds = BrainDataset(test, feature_cols, None, MAX_LEN)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

for fold, (train_idx, val_idx) in enumerate(kf.split(unique_samples)):
    print(f"\n=== Fold {fold+1}/{N_FOLDS} ===")
    
    train_ids, val_ids = unique_samples[train_idx], unique_samples[val_idx]
    train_ds = BrainDataset(train[train['sample_id'].isin(train_ids)], feature_cols, 'lever', MAX_LEN)
    val_ds = BrainDataset(train[train['sample_id'].isin(val_ids)], feature_cols, 'lever', MAX_LEN)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    
    # モデル初期化
    model = ClassicRecurrentModel(
        model_type=MODEL_TYPE,
        input_dim=len(feature_cols),
        hidden_dim=256,
        num_layers=3 # RNNは勾配消失しやすいので層を少し浅くするのがコツ
    ).to(DEVICE)
    
    if torch.cuda.device_count() > 1: model = nn.DataParallel(model)
        
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    
    best_loss = float('inf')
    best_model_path = f"best_{MODEL_TYPE.lower()}_fold{fold}.pth"
    
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        for batch in train_loader:
            x, y, m = batch['x'].to(DEVICE), batch['y'].to(DEVICE), batch['mask'].to(DEVICE)
            optimizer.zero_grad()
            preds = model(x)
            loss = (criterion(preds, y) * m).sum() / m.sum()
            loss.backward()
            # RNN系は勾配爆発対策が必須
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()
        scheduler.step()
        
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                x, y, m = batch['x'].to(DEVICE), batch['y'].to(DEVICE), batch['mask'].to(DEVICE)
                preds = model(x)
                loss = (criterion(preds, y) * m).sum() / m.sum()
                val_loss += loss.item()
        
        avg_val = val_loss / len(val_loader)
        if avg_val < best_loss:
            best_loss = avg_val
            torch.save(model.state_dict(), best_model_path)
            
        if (epoch+1) % 5 == 0:
            print(f"  Ep {epoch+1:2d} | Train: {train_loss/len(train_loader):.4f} | Val: {avg_val:.4f}")

    print(f"  >> Best Val: {best_loss:.5f}")

    # Inference
    try:
        model.load_state_dict(torch.load(best_model_path))
        model.eval()
        with torch.no_grad():
            for batch in test_loader:
                x = batch['x'].to(DEVICE)
                mask = batch['mask'].to(DEVICE)
                s_ids = batch['id']
                preds = model(x).cpu().numpy()
                mask = mask.cpu().numpy()
                for i, s_id in enumerate(s_ids):
                    v_len = int(mask[i].sum())
                    for t, val in enumerate(preds[i, :v_len]):
                        k = f"{s_id}_{t}"
                        final_test_preds[k] = final_test_preds.get(k, 0) + val
                        test_counts[k] = test_counts.get(k, 0) + 1
        
        # 中間保存
        temp_results = [{'id': k, 'lever': max(0, v / test_counts[k])} for k, v in final_test_preds.items()]
        pd.DataFrame(temp_results).to_csv(f'temp_submission_{MODEL_TYPE.lower()}_fold{fold+1}.csv', index=False)
        
    except Exception as e:
        print(f"  Error in Inference: {e}")

# ==============================================================================
# 6. Final Submission
# ==============================================================================
if len(final_test_preds) > 0:
    results = [{'id': k, 'lever': max(0, v / test_counts[k])} for k, v in final_test_preds.items()]
    df_sub = pd.DataFrame(results)
    df_sub.to_csv(SUBMISSION_PATH, index=False)
    print(f"\nSUCCESS: {SUBMISSION_PATH} created.")
else:
    print("ERROR: No predictions.")

>>> Running Mode: GRU
Using Device: cuda
>>> Loading & Preprocessing...

>>> Starting 5-Fold CV with GRU...

=== Fold 1/5 ===
  Ep  5 | Train: 0.8445 | Val: 1.0363
  Ep 10 | Train: 0.6597 | Val: 0.9167
  Ep 15 | Train: 0.4298 | Val: 0.9949
  Ep 20 | Train: 0.2492 | Val: 1.0001
  Ep 25 | Train: 0.1444 | Val: 1.0152
  Ep 30 | Train: 0.1192 | Val: 1.0247
  >> Best Val: 0.91672

=== Fold 2/5 ===
  Ep  5 | Train: 0.8824 | Val: 1.0092
  Ep 10 | Train: 0.6682 | Val: 0.8198
  Ep 15 | Train: 0.4189 | Val: 0.9150
  Ep 20 | Train: 0.2388 | Val: 0.9008
  Ep 25 | Train: 0.1416 | Val: 0.9401
  Ep 30 | Train: 0.1177 | Val: 0.9449
  >> Best Val: 0.81963

=== Fold 3/5 ===
  Ep  5 | Train: 0.8694 | Val: 0.9293
  Ep 10 | Train: 0.6497 | Val: 0.9973
  Ep 15 | Train: 0.3908 | Val: 0.9465
  Ep 20 | Train: 0.2206 | Val: 0.9358
  Ep 25 | Train: 0.1327 | Val: 1.0002
  Ep 30 | Train: 0.1102 | Val: 0.9943
  >> Best Val: 0.89088

=== Fold 4/5 ===
  Ep  5 | Train: 0.8687 | Val: 0.9025
  Ep 10 | Train: 0.6589 | Val